# Rubric Testing and Evaluation

This notebook demonstrates how to:
1. Test individual rubrics with sample data
2. Use the LangSmith wrapper for tracking rubric performance


In [1]:
#%load_ext autoreload
#%autoreload 2
import nest_asyncio
nest_asyncio.apply()

In [2]:
import pandas as pd

from explain.eval.rubrics import CorrectnessRubric, FalsificationRubric, PlausibilityRubric
from explain.eval.utils import as_verifier_dataset

df = pd.read_csv("../../data/curation_v1/results/structure-explain-results-v3-claude4.csv")
df["response"] = df["raw_response"].apply(lambda x: x.split("</think>")[1].strip())
# sort df by the length of the "dag" column
df = df.sort_values(by="dag", key=lambda x: x.str.len(), ascending=True)
dt = as_verifier_dataset(df, answer_columns=["response"])

/Users/emmanuel.noutahi/Code/hooke-explain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 176/176 [00:00<00:00, 7908.02 examples/s]


# Rubric Testing and Evaluation

Let's test the individual rubrics to understand their behavior.

In [3]:
example_dt = dt.select(range(10))
example_dt.column_names

['question', 'answer', 'info', 'task']

In [4]:
example_dt["answer"][0]

'<answer>\nOclacitinib competitively binds the ATP-binding pocket of JAK1 (and JAK2), directly inhibiting their catalytic activity. In the context of SOCS3 knockdown, which removes a key negative feedback regulator of JAK-STAT signaling, this creates competing effects on the pathway. SOCS3 deficiency normally leads to enhanced and prolonged STAT3 phosphorylation and nuclear translocation, resulting in upregulated inflammatory gene expression and cytokine production. However, Oclacitinib\'s direct enzymatic inhibition of JAK1 counteracts this hyperactivation by blocking upstream kinase activity. The net result is partial restoration of normal JAK-STAT signaling dynamics and reduced inflammatory responses compared to SOCS3 knockdown alone, creating a distinctive transcriptomic signature with suppressed STAT3-dependent gene expression and normalized cellular behaviors.\n</answer>\n\n<explain>\nset_context(cell_type="immune cell", disease_context="pathway biology modulation", prior_perturb

In [5]:
# correctness rubric
correctness_rubric = CorrectnessRubric()
correctness_rubric.judge(
    prompt=example_dt["question"][0],
    completion=example_dt["answer"][0],
    answer=example_dt["answer"][0],
    state={},
)

2025-09-04 14:26:21.412 | INFO     | explain.llm._client:__init__:587 - Initialized LiteLLM client with model claude-sonnet-4@20250514
2025-09-04 14:26:21.419 | INFO     | explain.llm._client:__init__:703 - Initialized unified LLM client with provider: litellm


{'correctness': 1.0}

In [6]:
# plausibility rubric
plausibility_rubric = PlausibilityRubric()
plausibility_rubric.judge(
    prompt=example_dt["question"][0],
    completion=example_dt["answer"][0],
    answer=example_dt["answer"][0],
    state={},
)

2025-09-04 14:26:24.606 | INFO     | explain.llm._client:__init__:587 - Initialized LiteLLM client with model claude-sonnet-4@20250514
2025-09-04 14:26:24.607 | INFO     | explain.llm._client:__init__:703 - Initialized unified LLM client with provider: litellm


{'scientific_accuracy': 0.9,
 'logical_consistency': 0.9,
 'mechanistic_clarity': 0.8}

In [7]:
from explain.eval.tools import LITERATURE_TOOLS, BIOLOGICAL_TOOLS, REGISTERED_TOOLS

In [8]:
REGISTERED_TOOLS

{'check_drug_target_interaction': <explain.eval.tools.bio.binding.DTIVerifier at 0x3648d0830>,
 'check_subcellular_translocation': <explain.eval.tools.bio.localization.SCLVerifier at 0x3641b51c0>,
 'chembl_chemical_resolver': <explain.eval.tools.chem.chembl.ChEMBLResolver at 0x3673aa180>,
 'pubchem_chemical_resolver': <explain.eval.tools.chem.pubchem.PubChemResolver at 0x364df4ef0>,
 'literature_fulltext_search': <explain.eval.tools.esearch.LitteratureSearcher at 0x367ec8950>,
 'literature_evidence_answerer': <explain.eval.tools.evidencer.Evidencer at 0x36bdf7230>}

In [9]:
BIOLOGICAL_TOOLS

{'check_drug_target_interaction': <explain.eval.tools.bio.binding.DTIVerifier at 0x364e13260>,
 'check_subcellular_translocation': <explain.eval.tools.bio.localization.SCLVerifier at 0x364e2a630>}

In [10]:
# falsification rubric
falsification_rubric = FalsificationRubric(llm_provider="litellm", tools=list(BIOLOGICAL_TOOLS.values()))
state = {}
falsification_rubric.judge(
    prompt=example_dt["question"][1],
    completion=example_dt["answer"][1],
    answer=example_dt["answer"][1],
    state=state,
)

2025-09-04 14:26:27.261 | INFO     | explain.llm._client:__init__:587 - Initialized LiteLLM client with model claude-sonnet-4@20250514
2025-09-04 14:26:27.262 | INFO     | explain.llm._client:__init__:703 - Initialized unified LLM client with provider: litellm
2025-09-04 14:26:30.423 | DEBUG    | explain.eval.tools.bio.binding:_tool_logic:43 - Args: drug='TG 101348' target='ENSG00000096968' interaction_type='inhibitor' strength=None
2025-09-04 14:26:34.714 | DEBUG    | explain.eval.tools.bio.binding:_tool_logic:65 - Target ID: O60674, Compound ID: JOOXLOJCABQBSG-UHFFFAOYSA-N
2025-09-04 14:27:10.259 | DEBUG    | explain.eval.tools.bio.binding:_tool_logic:68 - DTI Result: <explain.utils.bio._base_dti.DTIResult object at 0x3764beba0>
2025-09-04 14:27:10.260 | DEBUG    | explain.eval.tools.bio.binding:_tool_logic:90 - Feedback: {'target_id': 'O60674', 'compound_id': 'JOOXLOJCABQBSG-UHFFFAOYSA-N', 'score': 3.0, 'unit': 'nM', 'method': 'bioref', 'binding_type': 'inhibitor', 'confidence': 1.0

{'falsification_score': 1.0}

In [11]:
print(example_dt[1]["answer"])

<answer>
TG 101348 acts as an ATP-competitive inhibitor that binds directly to JAK2 with high selectivity, disrupting IL-13-mediated signaling cascades. Upon IL-13 binding to its receptor complex (IL13RA1/IL4R), JAK2 is normally activated to phosphorylate STAT6. TG 101348 mechanistically blocks this JAK2 activation, preventing STAT6 phosphorylation, dimerization, and nuclear translocation. This causal disruption of the JAK2-STAT6 pathway leads to decreased transcription of IL-13-responsive genes including mucins (Muc5b), goblet cell markers (Clca3, Clca1), and inflammatory mediators (Retnlb, Alox15). The functional outcome is a measurable rescue of asthmatic phenotypes, including reduced airway hyperresponsiveness, decreased mucus overproduction, and attenuated eosinophilic inflammation, effectively reversing the pathological features of IL-13-mediated asthma and allergic responses.
</answer>

<explain>
set_context(cell_type="airway epithelial cells", disease_context="IL-13-mediated as

In [12]:
state

{'judge_response': LLMResponse(content='<reason>\nLet me analyze each sub-claim from the explanation block:\n\n**Sub-claim n1: binds_to(TG 101348, JAK2, high selectivity, ATP-competitive binding)**\n- Tool evidence: check_drug_target_interaction confirmed TG 101348 binds to JAK2 (ENSG00000096968) as an inhibitor with 3.0 nM binding affinity and confidence 1.0\n- The high selectivity and ATP-competitive mechanism are consistent with known JAK2 inhibitor mechanisms\n- **SUPPORTED**\n\n**Sub-claim n2: modulates_molecule_activity(JAK2, down, ATP site occupancy)**\n- Tool evidence: The confirmed inhibitor binding with high affinity (3.0 nM) directly supports JAK2 activity inhibition\n- ATP-competitive inhibition is the standard mechanism for JAK2 inhibitors\n- **SUPPORTED**\n\n**Sub-claim n3: modulates_molecule_activity(STAT6, down, blocked phosphorylation by inhibited JAK2)**\n- This follows logically from JAK2 inhibition, as JAK2 is the primary kinase that phosphorylates STAT6 in IL-13 si